# Week 1: Financial Retail Data Collection and Preprocessing

**Organization:** CS Vision  
**Dataset:** Financial retail transactions

**Goal:** Understand the business data, clean the records, check the financial formulas, and create a clean dataset for simple reporting.

This notebook uses beginner-friendly Python, NumPy, and pandas only.

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

workspace_roots = [Path.cwd(), *Path.cwd().parents]
input_candidates = [
    root / 'Week-01_Data-Collection-Preprocessing' / 'data' / 'raw' / 'financial_retail_raw.csv'
    for root in workspace_roots
] + [
    root / 'data' / 'raw' / 'financial_retail_raw.csv'
    for root in workspace_roots
]
INPUT_FILE = next((path for path in input_candidates if path.is_file()), None)
if INPUT_FILE is None:
    raise FileNotFoundError('Could not find the Week 1 raw financial retail dataset.')
OUTPUT_FILE = INPUT_FILE.parent.parent / 'cleaned' / 'financial_retail_clean.csv'

## 1. Understand the financial business problem

Each row represents a retail transaction. The important business measures are revenue, cost, operating expense, gross profit, and net profit.

The two basic financial rules are:

- `Gross_Profit = Revenue - Cost`
- `Net_Profit = Gross_Profit - Operating_Expense`

In [24]:
# Dataset collection and loading
df_raw = pd.read_csv(INPUT_FILE)

print(f'Rows: {len(df_raw):,}')
print(f'Columns: {df_raw.shape[1]:,}')
display(df_raw.head())

# Data structure analysis and simple profiling
profile = pd.DataFrame({
    'data_type': df_raw.dtypes.astype(str),
    'missing_values': df_raw.isna().sum(),
    'unique_values': df_raw.nunique()
})
display(profile)
print(f'Exact duplicate rows: {df_raw.duplicated().sum():,}')

Rows: 865
Columns: 18


,Transaction_ID,Date,Revenue,Cost,Operating_Expense,Gross_Profit,Net_Profit,Product,Category,Customer_ID,Customer_Segment,Region,City,Quantity,Order_Status,Month,Quarter,Year
0,TXN100199,2025-09-25,"4,256.71","2,650.18",495.49,"1,606.53","1,111.04",Headphones,Accessories,CUST1200,Small Business,North,Jaipur,1,complete,September,Q3,2025
1,TXN100721,2025-01-05,"67,804.68","39,385.48","5,193.28","28,419.20","23,225.92",Monitor,Electronics,CUST1144,Small Business,South,bengaluru,4,complete,January,Q1,2025
2,TXN100636,2025-12-12,"100,619.61","69,694.03","9,672.72","30,925.58","21,252.86",Smartphone,Electronics,CUST1103,Consumer,West,Pune,4,Completed,December,Q4,2025
3,TXN100310,2025-06-07,"61,516.73","39,426.96","10,221.54","22,089.77","11,868.23",Printer,Electronics,CUST1108,Corporate,South,Chennai,4,Completed,June,Q2,2025
4,TXN100024,2025-06-10,"26,681.44","14,659.90","4,758.17","12,021.54","7,263.37",Headphones,Accessories,CUST1182,Small Business,South,bengaluru,6,Pending,June,Q2,2025


,data_type,missing_values,unique_values
Transaction_ID,str,0,850
Date,str,0,326
Revenue,float64,8,842
Cost,float64,0,850
Operating_Expense,float64,0,850
Gross_Profit,float64,0,850
Net_Profit,float64,0,851
Product,str,7,14
Category,str,0,5
Customer_ID,str,19,200


Exact duplicate rows: 13


## 2. Clean and transform the data

The steps below use pandas for duplicate removal, text cleanup, missing-value treatment, type conversion, sorting, and date operations. NumPy is used for simple numerical calculations and a conditional profit label.

In [22]:
# Remove exact duplicate records
df = df_raw.drop_duplicates().copy()

# Trim spaces in text columns
text_columns = [
    'Transaction_ID', 'Product', 'Category', 'Customer_ID',
    'Customer_Segment', 'Region', 'City', 'Order_Status'
]
for column in text_columns:
    df[column] = df[column].astype('string').str.strip()

# Use consistent text labels
df['Product'] = df['Product'].str.title()
df['Category'] = df['Category'].str.title()
df['Customer_Segment'] = df['Customer_Segment'].str.title()
df['City'] = df['City'].str.title()
df['Order_Status'] = df['Order_Status'].str.title().replace({'Complete': 'Completed'})

# Convert dates and financial fields to the correct data types
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
numeric_columns = [
    'Revenue', 'Cost', 'Operating_Expense',
    'Gross_Profit', 'Net_Profit', 'Quantity'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Fill missing values with simple labels or formula-based values
df['Product'] = df['Product'].fillna('Unknown')
df['Customer_ID'] = df['Customer_ID'].fillna('UNKNOWN')
df['City'] = df['City'].fillna('Unknown')
df['Order_Status'] = df['Order_Status'].fillna('Unknown')
df['Revenue'] = df['Revenue'].fillna(df['Cost'] + df['Gross_Profit'])

# Apply the financial formulas
df['Gross_Profit'] = (df['Revenue'] - df['Cost']).round(2)
df['Net_Profit'] = (df['Gross_Profit'] - df['Operating_Expense']).round(2)

# Create simple date fields from the Date column
df['Month'] = df['Date'].dt.month_name()
df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype('string')
df['Year'] = df['Date'].dt.year.astype('Int64')

# Simple filtering example: select completed transactions
completed_transactions = df[df['Order_Status'] == 'Completed']
print(f'Completed transactions: {len(completed_transactions):,}')

# NumPy numerical and conditional calculations
revenue_values = df['Revenue'].to_numpy()
print(f'Average revenue: {np.mean(revenue_values):,.2f}')
print(f'Total revenue: {np.sum(revenue_values):,.2f}')
df['Profit_Status'] = np.where(df['Net_Profit'] >= 0, 'Profit', 'Loss')

# Remove duplicates that appeared after text standardization
df = df.drop_duplicates()

# Sort the clean data by date
df = df.sort_values('Date').reset_index(drop=True)
display(df.head())

Completed transactions: 655
Average revenue: 71,902.36
Total revenue: 61,260,813.92


,Transaction_ID,Date,Revenue,Cost,Operating_Expense,Gross_Profit,Net_Profit,Product,Category,Customer_ID,Customer_Segment,Region,City,Quantity,Order_Status,Month,Quarter,Year,Profit_Status
0,TXN100447,2025-01-01,"58,470.21","36,725.89","7,621.28","21,744.32","14,123.04",Monitor,Electronics,CUST1087,Consumer,South,Chennai,4,Completed,January,Q1,2025,Profit
1,TXN100678,2025-01-01,"24,860.80","16,154.15","3,382.73","8,706.65","5,323.92",Smartphone,Electronics,CUST1189,Consumer,North,Delhi,1,Completed,January,Q1,2025,Profit
2,TXN100649,2025-01-01,"29,366.04","18,343.70","5,027.60","11,022.34","5,994.74",Smartphone,Electronics,CUST1163,Consumer,South,Chennai,1,Completed,January,Q1,2025,Profit
3,TXN100756,2025-01-01,"8,456.55","4,776.35","1,291.80","3,680.20","2,388.40",Smartwatch,Accessories,CUST1072,Corporate,West,Ahmedabad,1,Cancelled,January,Q1,2025,Profit
4,TXN100849,2025-01-01,"68,559.66","41,428.90","12,170.47","27,130.76","14,960.29",Laptop,Electronics,CUST1161,Consumer,North,Delhi,1,Completed,January,Q1,2025,Profit


## 3. Final data-quality and uniqueness validation

The data has already received basic preprocessing above. This final step checks transaction-ID uniqueness and removes only the specified erroneous `TXN100505` record with `Quantity = 75`. No other non-exact records are removed.

In [ ]:
# Final validation starts with the basic preprocessed DataFrame.
# Only exact duplicates and the specified TXN100505 outlier may be removed.
comparison_columns = [
    'Transaction_ID', 'Date', 'Revenue', 'Cost', 'Operating_Expense',
    'Gross_Profit', 'Net_Profit', 'Product', 'Category', 'Customer_ID',
    'Customer_Segment', 'Region', 'City', 'Quantity', 'Order_Status',
    'Month', 'Quarter', 'Year'
]
target_id = 'TXN100505'

rows_before = len(df)
duplicate_ids_before = int(df['Transaction_ID'].duplicated().sum())
exact_duplicates_before = int(df.duplicated().sum())

target_records = df[df['Transaction_ID'].eq(target_id)].copy()
print(f'{target_id} records before cleaning: {len(target_records)}')
display(target_records[comparison_columns].T)

# Confirm that the duplicate records match in every listed column except Quantity.
assert len(target_records) == 2
assert target_records[comparison_columns].drop(columns='Quantity').nunique().max() == 1
assert set(target_records['Quantity']) == {3, 75}

# Remove only the requested Quantity=75 record.
remove_mask = df['Transaction_ID'].eq(target_id) & df['Quantity'].eq(75)
assert int(remove_mask.sum()) == 1
df = df.loc[~remove_mask].copy()

rows_removed = rows_before - len(df)
duplicate_ids_after = int(df['Transaction_ID'].duplicated().sum())
exact_duplicates_after = int(df.duplicated().sum())

checks = {
    'rows_after_cleaning': len(df),
    'columns': df.shape[1],
    'duplicate_transaction_ids_after': duplicate_ids_after,
    'exact_duplicate_rows_after': exact_duplicates_after,
    'missing_cells': int(df.isna().sum().sum()),
    'negative_financial_values': int((df[numeric_columns] < 0).sum().sum()),
    'gross_profit_mismatches': int((~np.isclose(
        df['Gross_Profit'], df['Revenue'] - df['Cost'], atol=0.01
    )).sum()),
    'net_profit_mismatches': int((~np.isclose(
        df['Net_Profit'], df['Gross_Profit'] - df['Operating_Expense'], atol=0.01
    )).sum()),
    'month_mismatches': int((df['Date'].dt.month_name() != df['Month']).sum()),
    'quarter_mismatches': int((
        'Q' + df['Date'].dt.quarter.astype('string') != df['Quarter']
    ).sum()),
    'year_mismatches': int((
        df['Date'].dt.year.astype('Int64') != df['Year']
    ).sum())
}

display(pd.Series(checks, name='value'))

assert rows_removed == 1
assert duplicate_ids_after == 0
assert exact_duplicates_after == 0
assert checks['missing_cells'] == 0
assert checks['negative_financial_values'] == 0
assert checks['gross_profit_mismatches'] == 0
assert checks['net_profit_mismatches'] == 0
assert checks['month_mismatches'] == 0
assert checks['quarter_mismatches'] == 0
assert checks['year_mismatches'] == 0
assert df['Transaction_ID'].is_unique

# Export the final cleaned dataset.
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')

summary = pd.Series({
    'rows_before_cleaning': rows_before,
    'rows_removed': rows_removed,
    'rows_after_cleaning': len(df),
    'duplicate_transaction_ids_before': duplicate_ids_before,
    'duplicate_transaction_ids_after': duplicate_ids_after,
    'remaining_data_quality_issues': 'None'
}, name='value')
print(f'Saved final cleaned dataset to: {OUTPUT_FILE}')
display(summary)

# Simple final aggregation for reporting.
display(
    df.groupby('Category', observed=True)
      .agg(
          Transactions=('Transaction_ID', 'count'),
          Revenue=('Revenue', 'sum'),
          Net_Profit=('Net_Profit', 'sum')
      )
      .round(2)
)